In [ ]:
!pip install pybaseball

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
import os
from pybaseball import playerid_reverse_lookup
import gc # 垃圾回收，手動釋放 RAM

In [ ]:
# 掛載雲端硬碟
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/baseball data'

# 切換工作目錄到該資料夾
os.chdir(folder_path)
CURRENT_DIR = Path.cwd()

print(f"目前工作目錄已切換至：{CURRENT_DIR}")
FILES = ['statcast_2018.parquet', 'statcast_2019.parquet','statcast_2020.parquet','statcast_2021.parquet','statcast_2022.parquet','statcast_2023.parquet','statcast_2024.parquet']
OUT_PATH = CURRENT_DIR / 'clustering_batting_stats_2018_2024.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
目前工作目錄已切換至：/content/drive/.shortcut-targets-by-id/13DsmM8kGLjvpI-pfWi_vNHHPryTiQObp/baseball data


In [ ]:
if OUT_PATH.exists():
    df_batting = pd.read_csv(OUT_PATH)
    print(f'已載入快取：{len(df_batting)} 筆')
else:
    PA_EVENTS = {
        'single', 'double', 'triple', 'home_run', 'walk', 'intent_walk',
        'hit_by_pitch', 'strikeout', 'strikeout_double_play', 'field_out',
        'force_out', 'grounded_into_double_play', 'double_play', 'fielders_choice',
        'fielders_choice_out', 'field_error', 'sac_fly', 'sac_fly_double_play',
        'sac_bunt', 'sac_bunt_double_play', 'catcher_interf',
    }

    # 為了省 RAM，只讀取計算必要的欄位
    USE_COLS = ['batter', 'game_year', 'events', 'woba_value', 'woba_denom', 'game_pk']

    all_summaries = []

    for f in FILES:
        p = CURRENT_DIR / f
        if not p.exists():
            print(f'警告：找不到檔案 {f}')
            continue

        print(f'正在處理：{f} ...')
        # 1. 讀取單一年度資料並過濾
        sc_temp = pd.read_parquet(p, columns=USE_COLS)
        sc_temp = sc_temp[sc_temp['events'].isin(PA_EVENTS)]

        # 2. 進行該年度的 Groupby 計算
        rows = []
        for (batter_id, year), grp in sc_temp.groupby(['batter', 'game_year']):
            ev = grp['events']
            pa = len(grp)
            h1 = (ev == 'single').sum(); h2 = (ev == 'double').sum()
            h3 = (ev == 'triple').sum(); hr = (ev == 'home_run').sum()
            h  = h1 + h2 + h3 + hr
            bb = ev.isin(['walk', 'intent_walk']).sum()
            hbp= (ev == 'hit_by_pitch').sum()
            sf = ev.isin(['sac_fly', 'sac_fly_double_play']).sum()
            sb = ev.isin(['sac_bunt', 'sac_bunt_double_play']).sum()
            k  = ev.isin(['strikeout', 'strikeout_double_play']).sum()
            ab = pa - bb - hbp - sf - sb

            avg = h / ab if ab > 0 else 0
            obp = (h + bb + hbp) / (ab + bb + hbp + sf) if (ab + bb + hbp + sf) > 0 else 0
            slg = (h1 + 2*h2 + 3*h3 + 4*hr) / ab if ab > 0 else 0
            woba = grp['woba_value'].sum(skipna=True) / grp['woba_denom'].sum(skipna=True) if grp['woba_denom'].sum() > 0 else 0
            g = grp['game_pk'].nunique()

            rows.append({
                'batter': batter_id, 'Year': year, 'G': g, 'PA': pa, 'AB': int(ab),
                'H': int(h), '2B': int(h2), '3B': int(h3), 'HR': int(hr),
                'BB': int(bb), 'K': int(k), 'HBP': int(hbp), 'SF': int(sf),
                'AVG': round(avg, 3), 'OBP': round(obp, 3), 'SLG': round(slg, 3),
                'OPS': round(obp + slg, 3), 'ISO': round(slg - avg, 3),
                'BABIP': round((h - hr) / (ab - k - hr + sf) if (ab - k - hr + sf) > 0 else 0, 3),
                'BB%': round(bb / pa, 4), 'K%': round(k / pa, 4), 'wOBA': round(woba, 3)
            })

        # 3. 轉成小型的 DataFrame 並加入總表
        all_summaries.append(pd.DataFrame(rows))

        # 4. 關鍵：手動清空原始大資料並釋放記憶體
        del sc_temp
        gc.collect()

    # 合併所有年份的統計結果 (此時資料量很小，不會當機)
    df = pd.concat(all_summaries, axis=0).reset_index(drop=True)

    # 計算全聯盟基準 (用全 7 年平均)
    lg_woba = df['wOBA'].mean()
    woba_scale, lg_r_pa = 1.18, 0.127

    df['wRC+'] = df.apply(lambda r: round(((r['wOBA'] - lg_woba) / woba_scale / lg_r_pa + 1) * 100, 1) if r['PA'] > 0 else 100, axis=1)
    df['WAR'] = df.apply(lambda r: round(((r['wOBA'] - lg_woba) / woba_scale * r['PA'] + 20 * (r['PA'] / 600)) / 10, 1), axis=1)

    # 批量查找球員姓名 (一次查完省時間)
    unique_batters = df['batter'].unique().astype(int).tolist()
    ndf = playerid_reverse_lookup(unique_batters, key_type='mlbam')
    ndf['Name'] = ndf['name_first'].str.title() + ' ' + ndf['name_last'].str.title()
    ndf = ndf.rename(columns={'key_mlbam': 'batter'})

    df = df.merge(ndf[['batter', 'Name']], on='batter', how='left')
    df['Name'] = df['Name'].fillna('Unknown_' + df['batter'].astype(str))

    # 過濾與排序
    df_batting = df[df['PA'] >= 100].sort_values(['Year', 'wRC+'], ascending=[True, False]).reset_index(drop=True)

    col_order = ['batter', 'Name', 'Year', 'G', 'PA', 'AB', 'H', '2B', '3B', 'HR',
                'BB', 'K', 'HBP', 'SF', 'AVG', 'OBP', 'SLG', 'OPS',
                'ISO', 'BABIP', 'BB%', 'K%', 'wOBA', 'wRC+', 'WAR']
    df_batting = df_batting[[c for c in col_order if c in df_batting.columns]]
    df_batting.to_csv(OUT_PATH, index=False)

print(f'處理完成，共 {len(df_batting)} 筆資料，已輸出至 {OUT_PATH}')
display(df_batting.head())

正在處理：statcast_2018.parquet ...
正在處理：statcast_2019.parquet ...
正在處理：statcast_2020.parquet ...
正在處理：statcast_2021.parquet ...
正在處理：statcast_2022.parquet ...
正在處理：statcast_2023.parquet ...
正在處理：statcast_2024.parquet ...
Gathering player lookup table. This may take a moment.
處理完成，共 3120 筆資料，已輸出至 /content/drive/.shortcut-targets-by-id/13DsmM8kGLjvpI-pfWi_vNHHPryTiQObp/baseball data/clustering_batting_stats_2018_2024.csv


,batter,Name,Year,G,PA,AB,H,2B,3B,HR,...,OBP,SLG,OPS,ISO,BABIP,BB%,K%,wOBA,wRC+,WAR
0,545361,Mike Trout,2018,140,608,472,147,24,4,39,...,0.459,0.627,1.086,0.316,0.345,0.2007,0.2039,0.471,276.8,15.7
1,605141,Mookie Betts,2018,142,642,544,186,50,5,32,...,0.435,0.629,1.063,0.287,0.365,0.1324,0.1480,0.462,270.8,16.1
2,572228,Luke Voit,2018,52,182,160,50,5,1,15,...,0.396,0.638,1.033,0.325,0.365,0.1154,0.2692,0.451,263.5,4.4
3,502110,J. D. Martinez,2018,156,676,590,193,37,2,44,...,0.399,0.620,1.020,0.293,0.367,0.1080,0.2204,0.446,260.2,16.0
4,592885,Christian Yelich,2018,152,679,593,191,34,7,37,...,0.405,0.590,0.995,0.268,0.367,0.1134,0.2032,0.434,252.2,15.4


In [ ]:
# Verify required columns
required_cols = ['batter','Year','Name', 'G', 'PA', 'AVG', 'OBP', 'SLG', 'wRC+', 'WAR', 'BB%', 'K%', 'BABIP', 'ISO']
missing = [c for c in required_cols if c not in df_batting.columns]
print('缺少欄位：', missing)
df_batting.head()

缺少欄位： []


,batter,Name,Year,G,PA,AB,H,2B,3B,HR,...,OBP,SLG,OPS,ISO,BABIP,BB%,K%,wOBA,wRC+,WAR
0,545361,Mike Trout,2018,140,608,472,147,24,4,39,...,0.459,0.627,1.086,0.316,0.345,0.2007,0.2039,0.471,276.8,15.7
1,605141,Mookie Betts,2018,142,642,544,186,50,5,32,...,0.435,0.629,1.063,0.287,0.365,0.1324,0.1480,0.462,270.8,16.1
2,572228,Luke Voit,2018,52,182,160,50,5,1,15,...,0.396,0.638,1.033,0.325,0.365,0.1154,0.2692,0.451,263.5,4.4
3,502110,J. D. Martinez,2018,156,676,590,193,37,2,44,...,0.399,0.620,1.020,0.293,0.367,0.1080,0.2204,0.446,260.2,16.0
4,592885,Christian Yelich,2018,152,679,593,191,34,7,37,...,0.405,0.590,0.995,0.268,0.367,0.1134,0.2032,0.434,252.2,15.4


In [ ]:
# 標準化、聚類視覺化
# 範例code：找出 2024 年的替代者
from scipy.spatial.distance import cdist

def find_replacements(player_name, year, top_n=5):
    # 1. 找出目標球員在該年份的數據特徵與類別
    target = df_total[(df_total['Name'] == player_name) & (df_total['Year'] == year)]
    if target.empty: return "找不到該球員"

    target_cluster = target['Cluster'].values[0]
    target_features = target[features_scaled_columns] # 這是標準化後的特徵欄位

    # 2. 找出同年份、同類別的其他球員
    candidates = df_total[(df_total['Year'] == year) &
                          (df_total['Cluster'] == target_cluster) &
                          (df_total['Name'] != player_name)]

    # 3. 計算歐幾里得距離
    distances = cdist(target_features, candidates[features_scaled_columns], metric='euclidean')
    candidates = candidates.copy()
    candidates['distance'] = distances[0]

    # 4. 回傳距離最近的前 N 名
    return candidates.sort_values('distance').head(top_n)[['Name', 'wRC+', 'distance']]

# 範例：如果 2024 年 Mike Trout 受傷了，誰最像他？
# find_replacements('Mike Trout', '2024')